# Smoothing Without Lying: Savitzky–Golay and Derivatives
### 3.2 — Savitzky–Golay Smoothing and Derivatives

Smoothing is the most over-used and least-understood step in spectral data
processing. It is one function call — `savgol_filter(...)` — so it *feels* free.
It is not. Every smoothing choice trades **noise** against **resolution**, and
if you pick the wrong window you will quietly distort the very peaks you are
trying to measure.

So this notebook is **not** "how to call `scipy.signal.savgol_filter`." The goal
is to teach you how to *choose* the parameters — window length and polynomial
order — from things you can actually measure: your **peak width**, your **noise
level**, and your **point spacing**. By the end you'll be able to look at a
spectrum and estimate good smoothing settings before you touch the keyboard.

> **Note on ordering:** Lesson **3.1 (Noise, Signal, and Why Preprocessing
> Exists)** comes first and covers these ideas in depth; Section 1 below is a
> short self-contained recap. If you have done signal processing before, skim it.

**What we'll cover**

1. Recap: signal vs. noise, and what smoothing really does
2. A noisy spectrum with *known* peak widths (our ground truth)
3. Moving average — the simple baseline everyone already knows
4. Savitzky–Golay — what it actually does differently
5. Window length vs. polynomial order — the two knobs
6. Over- vs. under-smoothing, measured quantitatively
7. First derivative — baseline removal and finding peak centres
8. Second derivative — pulling apart overlapping bands
9. Why derivatives amplify noise (and what to do about it)
10. Practical rules of thumb
11. A small reusable helper

## 1. Setup and a quick recap of noise & smoothing

### The imports

We use the standard scientific Python stack plus our project's UV-Vis simulator.
The one new tool is `savgol_filter` from SciPy — the Savitzky–Golay filter.

In [ ]:
# Standard scientific Python stack -- nothing exotic.
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# The star of this notebook: the Savitzky-Golay filter.
from scipy.signal import savgol_filter

# Our project's UV-Vis simulator. Because the data is *simulated*, we know the
# true peak positions, widths, and noise level -- so we can check honestly
# whether smoothing helped or quietly distorted the signal.
from simulated_data import uvvis

# A folder for saved figures (regenerable scratch; git-ignored).
EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

# A consistent, readable plot style for the whole notebook.
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
print("Ready. SciPy savgol_filter imported.")

### What "noise" and "smoothing" actually mean

A measured spectrum is **signal + noise**:

- **Signal** — the chemistry: absorbance bands, peaks. These are *smooth* and
  *correlated* from point to point. A real band changes gradually; neighbouring
  points sit at similar values because they sample the same underlying shape.
- **Noise** — detector and electronic randomness. It is *high-frequency* and
  *uncorrelated*: each point jitters independently of its neighbours.

That difference is the whole game. **Smoothing is local averaging**: replace each
point with a value computed from its neighbours. Because the signal is similar
across neighbours, averaging barely changes it; because the noise is independent
across neighbours, averaging cancels much of it out.

But there is no free lunch. The wider you average, the more noise you remove —
and the more you blur genuine, sharp features. That is the **noise vs.
resolution trade-off**, and it runs through everything below.

### The two numbers that govern every choice

You only need two measurements to reason about smoothing:

- **FWHM** (full width at half maximum) — how wide your peaks are, in x-units
  (here, nm). This is the resolution you must *protect*.
- **Point spacing** Δx — the gap between adjacent x-axis points. Divide them and
  you get **points-per-FWHM**, the single most useful number for picking a
  window length. A window much wider than one FWHM will flatten the peak.

## 2. A noisy spectrum with known peak widths

Let's build a spectrum where we *know* the truth. We give `uvvis.simulate()`
explicit peaks as `(center_nm, FWHM_nm, amplitude)` triples, so the widths are
ground truth, not a guess. We include a **narrow** band (15 nm) and a **broad**
band (50 nm) so the effect of peak width on smoothing is visible.

We generate the data twice with the **same seed**: once with realistic noise,
and once with `noise=0` to get the underlying truth. Being able to compare
against the truth is the luxury of simulated data — it's how we'll *grade* every
method honestly.

In [ ]:
# Ground-truth peaks: (center_nm, FWHM_nm, amplitude_AU).
# One narrow band and one broad band, so width effects are obvious.
PEAKS = [
    (500.0, 15.0, 1.0),   # narrow band -- the one most at risk from over-smoothing
    (650.0, 50.0, 0.7),   # broad band  -- tolerates a wider window
]
NOISE = 0.02   # Gaussian noise sigma, in absorbance units
SEED = 11

# Noisy measurement and the noise-free truth share peaks, baseline, and seed.
noisy = uvvis.simulate(peaks=PEAKS, noise=NOISE, baseline=0.05, seed=SEED)
truth = uvvis.simulate(peaks=PEAKS, noise=0.0, baseline=0.05, seed=SEED)

wl = noisy.x              # wavelength axis (nm), shape (n_points,)
y_noisy = noisy.single()  # the measured spectrum, 1D
y_true = truth.single()   # the underlying truth, 1D

# Point spacing comes straight from the axis -- never assume it.
dx = wl[1] - wl[0]
print(f"axis: {noisy.x_label} ({noisy.x_unit}), {wl.size} points")
print(f"point spacing dx = {dx:.3f} nm")

# Points-per-FWHM: the key number for choosing a window.
print("\npoints per FWHM (the budget you have to work with):")
for center, fwhm, amp in PEAKS:
    print(f"  band @ {center:.0f} nm, FWHM {fwhm:.0f} nm  ->  {fwhm/dx:5.1f} points")

In [ ]:
# Plot the noisy measurement against the truth.
fig, ax = plt.subplots()
ax.plot(wl, y_noisy, lw=0.9, color="#9aa0a6", label="measured (noisy)")
ax.plot(wl, y_true, lw=2.0, color="#1a73e8", label="true signal (noise-free)")
ax.set_xlabel(f"{noisy.x_label} ({noisy.x_unit})")
ax.set_ylabel(noisy.y_label)
ax.set_title("Our test spectrum: a narrow band (500 nm) and a broad band (650 nm)")
ax.legend()
fig.tight_layout()
fig.savefig(EXPORTS / "01_noisy_vs_truth.png", dpi=150)
plt.show()

Notice the noise is roughly the same height everywhere, but it *hurts* the
narrow band far more than the broad one: the narrow peak has fewer points
describing it, so each noisy point matters more. This is exactly why narrow
features are the first thing you lose to careless smoothing.

## 3. The simple baseline: a moving average

Before reaching for anything fancy, let's smooth the way most people first
imagine it: a **moving average** (also called a boxcar). Replace each point with
the plain average of a window of neighbours. We implement it with
`np.convolve` and a uniform kernel.

A moving average is the honest baseline to beat — it shows *why* we need
something better.

In [ ]:
def moving_average(y, window):
    '''Simple boxcar moving average.

    Each output point is the unweighted mean of `window` neighbouring points.
    `window` should be odd so the average is centred on the point it replaces.
    'same' keeps the output the same length as the input.
    '''
    kernel = np.ones(window) / window      # uniform weights that sum to 1
    return np.convolve(y, kernel, mode="same")


# Try three window sizes, in points.
windows = [5, 15, 41]

fig, ax = plt.subplots()
ax.plot(wl, y_noisy, lw=0.7, color="#dadce0", label="noisy")
ax.plot(wl, y_true, lw=2.0, color="#1a73e8", label="truth")
for w in windows:
    ax.plot(wl, moving_average(y_noisy, w), lw=1.3, label=f"moving avg, w={w}")
ax.set_xlabel(f"{noisy.x_label} ({noisy.x_unit})")
ax.set_ylabel(noisy.y_label)
ax.set_title("Moving average: more smoothing also means more peak distortion")
ax.legend()
fig.tight_layout()
fig.savefig(EXPORTS / "02_moving_average.png", dpi=150)
plt.show()

Look at the narrow 500 nm band. As the window grows the noise calms down — but
the peak gets **shorter and wider**, drifting away from the true blue curve. A
boxcar treats the data inside the window as if it were *flat* (it just averages
it), so wherever the signal is curved — i.e. at every peak — it systematically
pulls the top down and fills the sides in. By `w=41` the narrow band is badly
mangled.

That's the problem Savitzky and Golay solved: **how do you average away noise
without assuming the signal is flat?**

## 4. Savitzky–Golay: fit a curve, don't just average

The Savitzky–Golay (Sav-Gol) filter does one clever thing. Inside each sliding
window it fits a **low-order polynomial** (a little parabola, say) by
least-squares, then uses that polynomial's value at the centre as the smoothed
point. It then slides the window along by one point and repeats.

Why this beats a moving average: a parabola can *follow the curvature* of a
peak. Where a boxcar assumes "flat" and clips the top, Sav-Gol's polynomial
bends with the band, so it removes noise while preserving peak **height** and
**position** far better.

It takes two parameters:

- `window_length` — how many points in each fit (must be **odd**, and larger
  than `polyorder`).
- `polyorder` — the degree of the polynomial (2 or 3 is typical).

In [ ]:
# Match the window so the comparison is fair: same width, different method.
W = 15           # window length in points (odd)
POLY = 3         # cubic polynomial inside each window

y_savgol = savgol_filter(y_noisy, window_length=W, polyorder=POLY)
y_boxcar = moving_average(y_noisy, W)

fig, ax = plt.subplots()
ax.plot(wl, y_noisy, lw=0.7, color="#dadce0", label="noisy")
ax.plot(wl, y_true, lw=2.2, color="#1a73e8", label="truth")
ax.plot(wl, y_boxcar, lw=1.4, color="#ea4335", label=f"moving avg (w={W})")
ax.plot(wl, y_savgol, lw=1.6, color="#188038", label=f"Sav-Gol (w={W}, poly={POLY})")
ax.set_xlim(460, 540)   # zoom on the narrow band, where the difference shows
ax.set_xlabel(f"{noisy.x_label} ({noisy.x_unit})")
ax.set_ylabel(noisy.y_label)
ax.set_title("Same window, two methods: Sav-Gol keeps the peak height")
ax.legend()
fig.tight_layout()
fig.savefig(EXPORTS / "03_savgol_vs_boxcar.png", dpi=150)
plt.show()

# Quantify it: how much of the true peak height did each method keep?
i_peak = np.argmin(np.abs(wl - 500))         # index nearest 500 nm
true_h = y_true[i_peak]
print(f"true peak height           : {true_h:.3f} AU")
print(f"moving-average peak height : {y_boxcar[i_peak]:.3f} AU  "
      f"({100*y_boxcar[i_peak]/true_h:.0f}% retained)")
print(f"Sav-Gol peak height        : {y_savgol[i_peak]:.3f} AU  "
      f"({100*y_savgol[i_peak]/true_h:.0f}% retained)")

At the *same* window width, the moving average has visibly chopped the narrow
band down, while Sav-Gol sits almost on top of the truth. That height retention
is the practical reason Sav-Gol is the default smoother in spectroscopy: it
suppresses noise without quietly shrinking your analyte signal.

## 5. The two knobs: window length vs. polynomial order

These two parameters pull in opposite directions, and understanding the tug is
how you tune Sav-Gol with intent rather than trial-and-error:

- **Window length** (bigger = more smoothing). More points per fit means more
  noise averaged away — but a window much wider than the peak forces the
  polynomial to span features it shouldn't, distorting them.
- **Polynomial order** (higher = more flexible, *less* smoothing). A higher
  degree can follow sharper curvature, so it distorts peaks less — but it also
  hugs the noise more closely, so it smooths less for a given window.

Rules baked into the function: the window must be **odd** and **strictly greater
than** the polyorder. Below we sweep both so you can see the interaction.

In [ ]:
window_grid = [9, 21, 41]
poly_grid = [2, 4]

fig, axes = plt.subplots(len(poly_grid), len(window_grid),
                         figsize=(12, 6), sharex=True, sharey=True)
for r, poly in enumerate(poly_grid):
    for c, win in enumerate(window_grid):
        ax = axes[r, c]
        ys = savgol_filter(y_noisy, window_length=win, polyorder=poly)
        ax.plot(wl, y_noisy, lw=0.5, color="#e8eaed")
        ax.plot(wl, y_true, lw=1.6, color="#1a73e8")
        ax.plot(wl, ys, lw=1.4, color="#188038")
        ax.set_xlim(460, 540)            # focus on the fragile narrow band
        ax.set_title(f"window={win}, poly={poly}", fontsize=10)
# Shared labels.
for ax in axes[-1, :]:
    ax.set_xlabel(f"{noisy.x_label} ({noisy.x_unit})")
for ax in axes[:, 0]:
    ax.set_ylabel(noisy.y_label)
fig.suptitle("Window vs. polyorder (blue = truth, green = Sav-Gol). "
             "Big window + low order = most peak distortion", y=1.02)
fig.tight_layout()
fig.savefig(EXPORTS / "04_window_vs_poly_grid.png", dpi=150, bbox_inches="tight")
plt.show()

Read the grid left-to-right (wider window) and top-to-bottom (higher order). The
worst distortion is the **bottom-left-to-right** drift at low order with a large
window — the parabola simply can't bend enough to follow a narrow peak across
41 points, so it clips it. Raising the order to 4 (bottom row) recovers a lot of
that height at the same window, because a quartic can follow the band's
curvature. The trade: that flexibility means it also leaves a little more noise.

## 6. Over- vs. under-smoothing, measured

Eyeballing only gets you so far. Let's put numbers on the trade-off. We sweep
the window length and, for each, measure two competing quantities:

- **Residual noise** — the standard deviation of (smoothed − truth) in a flat,
  peak-free region (here 400–450 nm). Lower is better; it falls as the window
  grows.
- **Peak height retained** — the smoothed height of the narrow 500 nm band as a
  percentage of its true height. Higher is better; it falls as the window grows
  and starts eating the peak.

Plotting both against window length makes the **sweet spot** visible: smooth
enough to kill noise, not so much that you crush the peak.

In [ ]:
# A flat region with no peaks, for measuring leftover noise.
flat = (wl >= 400) & (wl <= 450)
i_peak = np.argmin(np.abs(wl - 500))
true_h = y_true[i_peak]

odd_windows = np.arange(5, 81, 2)   # odd windows from 5 to 79 points
resid_noise = []
height_kept = []
for win in odd_windows:
    ys = savgol_filter(y_noisy, window_length=win, polyorder=3)
    resid_noise.append(np.std(ys[flat] - y_true[flat]))
    height_kept.append(100 * ys[i_peak] / true_h)

resid_noise = np.array(resid_noise)
height_kept = np.array(height_kept)

# Express the narrow band's FWHM in points, to mark it on the plot.
fwhm_pts = 15.0 / dx

fig, ax1 = plt.subplots()
ax1.plot(odd_windows, resid_noise, "o-", color="#ea4335",
         label="residual noise (lower=better)")
ax1.set_xlabel("Sav-Gol window length (points)")
ax1.set_ylabel("residual noise (AU)", color="#ea4335")
ax1.tick_params(axis="y", labelcolor="#ea4335")

ax2 = ax1.twinx()
ax2.plot(odd_windows, height_kept, "s-", color="#188038",
         label="peak height retained (%)")
ax2.set_ylabel("narrow-peak height retained (%)", color="#188038")
ax2.tick_params(axis="y", labelcolor="#188038")
ax2.grid(False)

# Mark "window = one FWHM", a good rule-of-thumb target.
ax1.axvline(fwhm_pts, color="#5f6368", ls="--", lw=1)
ax1.text(fwhm_pts + 1, ax1.get_ylim()[1]*0.9,
         f"window ~ 1 FWHM\n({fwhm_pts:.0f} pts)", fontsize=9, color="#5f6368")

ax1.set_title("The smoothing trade-off: noise vs. peak height (narrow band)")
fig.tight_layout()
fig.savefig(EXPORTS / "05_tradeoff_curves.png", dpi=150)
plt.show()

print(f"Narrow band FWHM in points: {fwhm_pts:.0f}")
print("Under-smoothing  = small window: noise still high.")
print("Over-smoothing   = large window: peak height collapses.")
print("Sweet spot is near window ~ 1 FWHM, where noise is already low but the")
print("peak is still ~intact.")

The red curve falls fast then flattens: past a point, extra width buys almost no
noise reduction. The green curve is flat at first, then drops off a cliff once
the window outgrows the peak. The **sweet spot is the elbow** — right around
*window ≈ one FWHM in points*. That's the rule of thumb worth remembering: size
your window to your narrowest peak, not to how clean you wish the data looked.

## 7. The first derivative: removing baselines and finding centres

Sav-Gol can do more than smooth — it can compute a **smoothed derivative** in
the same pass (`deriv=1`). Derivatives are a workhorse of spectroscopy because:

- A **constant or sloping baseline** vanishes when you differentiate (the
  derivative of a constant is zero; of a straight line, a constant). This is why
  derivative spectra are far less sensitive to instrument drift.
- The first derivative **crosses zero** exactly at a peak's maximum (where the
  slope flips from positive to negative). That's a baseline-free way to locate a
  peak centre.

One required extra argument: `delta`, the spacing between points, so the
derivative comes out in the right units (per nm). Pass `delta=dx`.

In [ ]:
# Smooth first derivative via Sav-Gol. delta=dx gives units of AU per nm.
d1 = savgol_filter(y_noisy, window_length=15, polyorder=3, deriv=1, delta=dx)

fig, (axA, axB) = plt.subplots(2, 1, sharex=True, figsize=(9, 6))
axA.plot(wl, y_noisy, lw=0.7, color="#dadce0", label="noisy")
axA.plot(wl, savgol_filter(y_noisy, 15, 3), lw=1.6, color="#188038",
         label="smoothed")
axA.set_ylabel(noisy.y_label)
axA.set_title("Spectrum (top) and its first derivative (bottom)")
axA.legend()

axB.axhline(0, color="#5f6368", lw=1)
axB.plot(wl, d1, lw=1.5, color="#a142f4", label="1st derivative")
# The zero-crossing near 500 nm marks the narrow peak's centre.
axB.set_xlabel(f"{noisy.x_label} ({noisy.x_unit})")
axB.set_ylabel("dA/dλ")
axB.legend()

# Find the zero-crossing near the 500 nm band: the point where the derivative
# flips from positive (rising edge) to negative (falling edge). Detecting a
# genuine sign change is far more robust than just looking for |d1| ~ 0, which
# noise near the window edges can fool.
search = (wl > 480) & (wl < 520)
idx = np.where(search)[0]
sign_flip = np.where(np.diff(np.sign(d1[idx])) < 0)[0]   # + -> - crossing
zc = idx[sign_flip[0]]
axB.axvline(wl[zc], color="#5f6368", ls="--", lw=1)
axA.axvline(wl[zc], color="#5f6368", ls="--", lw=1)
axB.text(wl[zc] + 2, axB.get_ylim()[1]*0.6,
         f"zero-crossing\n= {wl[zc]:.0f} nm", fontsize=9)
fig.tight_layout()
fig.savefig(EXPORTS / "06_first_derivative.png", dpi=150)
plt.show()

print(f"True narrow-band centre   : 500 nm")
print(f"1st-derivative zero-cross : {wl[zc]:.0f} nm  (baseline-independent!)")

## 8. The second derivative: resolving overlapping bands

Here is where derivatives earn their keep. Real spectra are full of **overlapping
bands** that blur into a single lump — you can see "something is there" but not
how many peaks or where. The **second derivative** sharpens them.

The reason: differentiating twice exaggerates curvature. Each underlying peak
produces a sharp **negative minimum** in the second derivative, located right at
that peak's centre — even when the peaks overlap so much that the raw spectrum
shows only one bump. (Note the convention: an upward peak shows up as a
*downward* trough in the 2nd derivative.)

Let's make two heavily overlapping bands and pull them apart.

In [ ]:
# Two bands only 35 nm apart, each 40 nm FWHM -> they overlap heavily.
OVERLAP_PEAKS = [
    (600.0, 40.0, 1.0),
    (635.0, 40.0, 0.9),
]
ov = uvvis.simulate(peaks=OVERLAP_PEAKS, noise=0.01, baseline=0.1, seed=3)
ov_truth = uvvis.simulate(peaks=OVERLAP_PEAKS, noise=0.0, baseline=0.1, seed=3)
y_ov = ov.single()
wl_ov = ov.x
dx_ov = wl_ov[1] - wl_ov[0]

# Smooth + 2nd derivative in one Sav-Gol pass.
d2 = savgol_filter(y_ov, window_length=21, polyorder=3, deriv=2, delta=dx_ov)

fig, (axA, axB) = plt.subplots(2, 1, sharex=True, figsize=(9, 6))
axA.plot(wl_ov, y_ov, lw=1.4, color="#1a73e8", label="measured (looks like ONE peak)")
axA.set_ylabel(ov.y_label)
axA.set_title("Two overlapping bands hidden in one bump -- revealed by the 2nd derivative")
axA.legend()

axB.axhline(0, color="#5f6368", lw=1)
axB.plot(wl_ov, d2, lw=1.5, color="#e8710a", label="2nd derivative")
axB.set_xlabel(f"{ov.x_label} ({ov.x_unit})")
axB.set_ylabel("d²A/dλ²")
axB.legend()

# Locate the troughs (the band centres). A second-derivative minimum is a
# maximum of -d2, so we let scipy.signal.find_peaks do the work. The
# `prominence` filter keeps only troughs that stand out clearly, so leftover
# noise wiggles don't get mistaken for bands.
from scipy.signal import find_peaks
region = (wl_ov > 560) & (wl_ov < 680)            # search around the lump
neg = -d2
troughs, _ = find_peaks(neg, prominence=neg[region].max() * 0.3)
troughs = [t for t in troughs if region[t]]       # keep those in the region
for m in troughs:
    axB.axvline(wl_ov[m], color="#5f6368", ls="--", lw=1)
    axA.axvline(wl_ov[m], color="#5f6368", ls="--", lw=1)
fig.tight_layout()
fig.savefig(EXPORTS / "07_second_derivative.png", dpi=150)
plt.show()

print("True centres     : 600 nm and 635 nm")
print(f"2nd-deriv minima : {', '.join(f'{wl_ov[m]:.0f} nm' for m in troughs)}")

The top panel looks like a single broad feature — most people would report one
peak. The second derivative resolves it into **two clear troughs** sitting right
on the true 600 and 635 nm centres. This is a standard trick for counting and
locating bands in IR, UV-Vis, and NIR. But notice we had to smooth fairly
aggressively (`window=21`) to get a clean result — which brings us to the catch.

## 9. The catch: derivatives amplify noise

Differentiation is a noise amplifier. Intuitively, a derivative measures
*point-to-point change* — and noise is *nothing but* point-to-point change. Each
time you differentiate, high-frequency noise gets multiplied relative to the
smooth signal. A second derivative amplifies it twice over.

Watch what happens if we take the 2nd derivative with **too little smoothing**
versus **adequate smoothing**:

In [ ]:
fig, (axA, axB) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

# Under-smoothed: small window -> noise explodes in the 2nd derivative.
d2_under = savgol_filter(y_ov, window_length=7, polyorder=3, deriv=2, delta=dx_ov)
axA.axhline(0, color="#5f6368", lw=1)
axA.plot(wl_ov, d2_under, lw=1.0, color="#ea4335")
axA.set_title("2nd derivative, window=7  (too little smoothing: noise wins)")
axA.set_xlabel(f"{ov.x_label} ({ov.x_unit})")
axA.set_ylabel("d²A/dλ²")

# Adequately smoothed: the two troughs are clean and readable.
d2_ok = savgol_filter(y_ov, window_length=21, polyorder=3, deriv=2, delta=dx_ov)
axB.axhline(0, color="#5f6368", lw=1)
axB.plot(wl_ov, d2_ok, lw=1.4, color="#188038")
axB.set_title("2nd derivative, window=21  (enough smoothing: signal wins)")
axB.set_xlabel(f"{ov.x_label} ({ov.x_unit})")
fig.tight_layout()
fig.savefig(EXPORTS / "08_deriv_noise.png", dpi=150)
plt.show()

Same data, same operation — only the window changed. On the left the noise
completely swamps the two bands we were trying to find; on the right they're
obvious. The lesson:

> **Higher derivatives need more smoothing.** Smoothing and differentiation are
> not separate steps to argue about — Sav-Gol does both in one pass, and you
> should lean on it. As a rule, a second derivative wants a noticeably larger
> window than you'd use for plain smoothing, and `polyorder` must be at least
> `deriv + 1` (you can't get a 2nd derivative out of a straight line).

## 10. Practical rules of thumb

A pocket reference you can apply at the bench:

- **Measure first.** Get your narrowest peak's FWHM and your point spacing Δx.
  Their ratio (points-per-FWHM) drives everything.
- **Window length ≈ one FWHM in points** for plain smoothing. Bigger crushes
  peaks; smaller barely helps. Always **odd**.
- **Polyorder 2 or 3** for smoothing. Higher orders distort less but smooth
  less. For a derivative, `polyorder ≥ deriv + 1`.
- **You need resolution to begin with.** If a peak is described by fewer than
  ~6–10 points, no smoothing will save it — fix the acquisition (smaller step,
  slower scan) instead.
- **Smoothing and differentiating go together.** Use `savgol_filter(..., deriv=n,
  delta=dx)` in one call rather than differentiating raw data.
- **Higher derivative ⇒ larger window.** 2nd derivatives amplify noise hard;
  give them room.
- **Always overlay the result on the raw data.** If the smoothed curve pulls
  away from peak tops, you've over-smoothed. Trust your eyes plus the
  height-retention check from Section 6.
- **Report your parameters.** "Sav-Gol, window 15, polyorder 3" belongs in your
  methods — smoothing is data processing, and reproducibility demands it.

## 11. A small reusable helper

Wrapping the choices into two tiny functions (with sane defaults and clear
comments) makes them easy to reuse in later notebooks — and forces the important
parameters out into the open where you'll remember to report them.

In [ ]:
def smooth(y, fwhm_points, polyorder=3):
    '''Sav-Gol smooth, sizing the window to the peak width.

    Pass the narrowest peak's FWHM *in points*; the window is set to about one
    FWHM and forced odd. This bakes in the Section-6 rule of thumb.
    '''
    win = int(round(fwhm_points))
    if win % 2 == 0:               # window_length must be odd
        win += 1
    win = max(win, polyorder + 2)  # and strictly greater than polyorder
    return savgol_filter(y, window_length=win, polyorder=polyorder)


def derivative(y, dx, order=1, fwhm_points=15, polyorder=3):
    '''Smoothed Sav-Gol derivative.

    `order` is the derivative (1 or 2). Window is widened for higher orders,
    since derivatives amplify noise; polyorder is bumped up if needed so that
    polyorder >= order + 1. `dx` gives the result physical units.
    '''
    polyorder = max(polyorder, order + 1)
    win = int(round(fwhm_points * (1 + order)))  # more smoothing for higher orders
    if win % 2 == 0:
        win += 1
    win = max(win, polyorder + 2)
    return savgol_filter(y, window_length=win, polyorder=polyorder,
                         deriv=order, delta=dx)


# Quick sanity check against the spectrum from Section 2.
sm = smooth(y_noisy, fwhm_points=15.0 / dx)
d1 = derivative(y_noisy, dx, order=1, fwhm_points=15.0 / dx)
print("helper smooth() output shape   :", sm.shape)
print("helper derivative() output shape:", d1.shape)

fig, ax = plt.subplots()
ax.plot(wl, y_noisy, lw=0.7, color="#dadce0", label="noisy")
ax.plot(wl, sm, lw=1.6, color="#188038", label="smooth() helper")
ax.set_xlabel(f"{noisy.x_label} ({noisy.x_unit})")
ax.set_ylabel(noisy.y_label)
ax.set_title("The reusable helper, applied")
ax.legend()
fig.tight_layout()
plt.show()

## Key Takeaways

- **Smoothing trades noise against resolution** — it is a deliberate choice, not
  a free default.
- **Two numbers drive every decision:** the peak FWHM and the point spacing Δx,
  through their ratio (points-per-FWHM).
- **Savitzky–Golay fits a local polynomial**, so it preserves peak height and
  position far better than a moving average (98% vs. 81% in our test).
- **Window ≈ one FWHM in points is the sweet spot** — bigger crushes peaks,
  smaller barely helps.
- **Derivatives remove baselines and resolve overlap** (1st-derivative
  zero-crossing = centre; 2nd-derivative minima = overlapping centres) — but
  they amplify noise, so higher derivatives need more smoothing.

## Practical Checklist

- [ ] Measure the narrowest FWHM and Δx; compute points-per-FWHM.
- [ ] `window_length` **odd**, ≈ one FWHM in points, and **>** `polyorder`.
- [ ] `polyorder` 2–3 for smoothing; `polyorder ≥ deriv + 1` for derivatives.
- [ ] Make sure each peak has ≥ ~6–10 points before smoothing — otherwise fix the
  acquisition, not the data.
- [ ] Compute derivatives in one pass: `savgol_filter(..., deriv=n, delta=dx)`.
- [ ] Overlay the result on the raw data and check peak-height retention.

## Common Mistakes

- **Over-smoothing:** a window much wider than the peak flattens narrow bands and
  biases heights.
- **Under-smoothing a derivative:** 2nd-derivative noise swamps the bands.
- Forgetting `delta=dx`, so the derivative comes out in the wrong units.
- `polyorder` too low for the derivative order (you can't get a 2nd derivative
  out of a straight line).
- Treating smoothing as free — and not recording it.

## Reporting Guidance

- Report the exact filter settings in your methods: **"Savitzky–Golay, window N
  points, polyorder P (deriv = d)."** Smoothing is data processing, and
  reproducibility demands that it be stated.

## Next Lesson

**3.3 — Baseline Correction Fundamentals.** You can now smooth and differentiate
deliberately; next we tackle the sloping, drifting backgrounds that derivatives
only sidestep.